# Understanding Attention — From Scratch to FlexAttention

**What you'll learn in this notebook:**
- The intuition behind attention as a soft lookup table
- Scaled dot-product attention math (Q, K, V) step by step
- Why we scale by sqrt(d_k) — with numerical evidence
- Causal masking for autoregressive models
- Multi-head attention: motivation and implementation
- PyTorch's optimized `F.scaled_dot_product_attention` (SDPA)
- Building a complete Transformer block
- Positional encodings: why and how
- FlexAttention API patterns

**Prerequisites:** Matrix multiplication, softmax, basic neural network concepts.

**All code runs on CPU — no GPU required.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
print(f"PyTorch version: {torch.__version__}")

## Intuition: Attention as a Soft Lookup Table

Think of attention like a dictionary lookup, but *soft* (differentiable):
- **Keys (K):** What information is available (the "index")
- **Values (V):** The actual content stored at each key
- **Query (Q):** What you're looking for

A hard lookup finds the single best-matching key. Attention computes a *weighted average* of all values, where weights come from query-key similarity.

In [ ]:
# Intuition demo: soft lookup vs hard lookup
torch.manual_seed(42)

# Imagine 4 "memory slots" with 3-dimensional keys and values
keys = torch.tensor([[1.0, 0, 0],    # slot 0: "red" key
                     [0.0, 1, 0],    # slot 1: "green" key
                     [0.0, 0, 1],    # slot 2: "blue" key
                     [0.5, 0.5, 0]]) # slot 3: "yellow" key
values = torch.tensor([[10.0], [20.0], [30.0], [40.0]])  # stored values

# Query: "something between red and yellow"
query = torch.tensor([[0.8, 0.3, 0.0]])

# Compute similarity (dot product)
similarities = query @ keys.T
print(f"Similarities: {similarities}")

# Hard lookup: pick the max
hard_result = values[similarities.argmax()]
print(f"Hard lookup result: {hard_result.item():.1f} (just picks the best match)")

# Soft lookup: weighted average via softmax
weights = F.softmax(similarities, dim=-1)
soft_result = weights @ values
print(f"Soft lookup weights: {weights.squeeze()}")
print(f"Soft lookup result: {soft_result.item():.1f} (blends all values by similarity)")

## Scaled Dot-Product Attention — Step by Step

The attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ has shape `[seq_len_q, d_k]` — what each position is looking for
- $K$ has shape `[seq_len_k, d_k]` — what each position offers as a key
- $V$ has shape `[seq_len_k, d_v]` — the content at each position
- $d_k$ is the key dimension (used for scaling)

Let's implement this step by step with explicit shape tracking.

In [ ]:
torch.manual_seed(0)

# Dimensions
batch_size = 2
seq_len = 4   # sequence length (same for Q, K, V in self-attention)
d_k = 8       # key/query dimension
d_v = 8       # value dimension

# Create Q, K, V (in practice, these come from linear projections of input)
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

print(f"Q shape: {Q.shape}  (batch, seq_len_q, d_k)")
print(f"K shape: {K.shape}  (batch, seq_len_k, d_k)")
print(f"V shape: {V.shape}  (batch, seq_len_k, d_v)")

# Step 1: Compute attention scores (Q @ K^T)
scores = torch.bmm(Q, K.transpose(1, 2))  # [batch, seq_q, seq_k]
print(f"\nStep 1 — scores = Q @ K^T: {scores.shape}")
print(f"scores[0]:\n{scores[0]}")

# Step 2: Scale by sqrt(d_k)
scale = math.sqrt(d_k)
scaled_scores = scores / scale
print(f"\nStep 2 — scaled (÷ sqrt({d_k}) = {scale:.2f}):\n{scaled_scores[0]}")

# Step 3: Softmax over the key dimension
attn_weights = F.softmax(scaled_scores, dim=-1)  # [batch, seq_q, seq_k]
print(f"\nStep 3 — attention weights (softmax over keys):\n{attn_weights[0]}")
print(f"Each row sums to: {attn_weights[0].sum(dim=-1)}")

# Step 4: Weighted sum of values
output = torch.bmm(attn_weights, V)  # [batch, seq_q, d_v]
print(f"\nStep 4 — output = weights @ V: {output.shape}")

## Why Scale by sqrt(d_k)?

Without scaling, as `d_k` grows, dot products get larger → softmax becomes more peaked (nearly one-hot) → gradients vanish. Let's demonstrate:

In [ ]:
# Show the effect of scaling on attention sharpness
for d_k in [8, 64, 512]:
    q = torch.randn(1, 1, d_k)
    k = torch.randn(1, 10, d_k)
    
    # Without scaling
    raw_scores = torch.bmm(q, k.transpose(1, 2))
    raw_attn = F.softmax(raw_scores, dim=-1)
    
    # With scaling
    scaled_scores = raw_scores / math.sqrt(d_k)
    scaled_attn = F.softmax(scaled_scores, dim=-1)
    
    print(f"d_k={d_k:3d} | raw scores std={raw_scores.std():.2f} | "
          f"max attn weight: unscaled={raw_attn.max():.4f}, scaled={scaled_attn.max():.4f}")

print("\nWithout scaling, higher d_k → sharper attention → vanishing gradients!")
print("Scaling keeps the variance of scores ≈ 1 regardless of d_k.")

## Causal Masking for Autoregressive Models

In language models, position $i$ should only attend to positions $\leq i$ (can't look into the future). We achieve this with a **causal mask** — setting future positions to $-\infty$ before softmax.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Full attention implementation with optional mask."""
    d_k = Q.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.bmm(attn_weights, V)
    return output, attn_weights


seq_len = 5

# Create causal mask: lower triangular (position i can attend to positions <= i)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # [1, seq, seq]

print("Causal mask (1=attend, 0=block):")
print(causal_mask[0].int())

# Apply to random Q, K, V
Q = torch.randn(1, seq_len, 16)
K = torch.randn(1, seq_len, 16)
V = torch.randn(1, seq_len, 16)

output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print(f"\nCausal attention weights (notice upper triangle is 0):")
print(weights[0].detach())
print(f"\nRow 0 only attends to position 0: {weights[0, 0]}")
print(f"Row 4 attends to all positions: {weights[0, 4]}")

In [ ]:
# Verify: different heads learn different patterns
torch.manual_seed(123)
d_model, num_heads = 32, 4
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(1, 8, d_model)
_, weights = mha(x)

print("Attention weights per head (position 4 attending to all positions):")
for h in range(num_heads):
    w = weights[0, h, 4, :].detach()
    print(f"  Head {h}: {w.numpy().round(3)}")
print("\nNotice: each head attends to different positions!")

## Multi-Head Attention

Why multiple heads? A single attention head can only compute one type of relationship. Multiple heads let the model jointly attend to information from different representation subspaces (e.g., one head learns positional patterns, another learns semantic similarity).

Implementation: split `d_model` into `num_heads` separate attention computations, each operating on `d_k = d_model // num_heads` dimensions.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project and reshape: [batch, seq, d_model] → [batch, heads, seq, d_k]
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Attention per head: [batch, heads, seq, d_k] @ [batch, heads, d_k, seq]
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = attn_weights @ V  # [batch, heads, seq, d_k]
        
        # Concatenate heads: [batch, heads, seq, d_k] → [batch, seq, d_model]
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attn_output), attn_weights


# Test it
d_model, num_heads = 64, 8
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 10, d_model)  # [batch=2, seq=10, d_model=64]
output, weights = mha(x)

print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights: {weights.shape} (batch, heads, seq_q, seq_k)")
print(f"\nEach head has d_k = {d_model // num_heads}")
print(f"Total params: {sum(p.numel() for p in mha.parameters()):,}")

## PyTorch's F.scaled_dot_product_attention (SDPA)

PyTorch provides a highly optimized attention implementation that automatically selects the best backend (Flash Attention, Memory-Efficient Attention, or Math fallback). Use this in production — it's faster and more memory-efficient than manual implementations.

In [ ]:
# SDPA expects shape: [batch, num_heads, seq_len, d_k]
batch, heads, seq_len, d_k = 2, 8, 16, 32

Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

# Basic usage — automatically picks best backend
output = F.scaled_dot_product_attention(Q, K, V)
print(f"SDPA output: {output.shape}")

# With causal mask (is_causal=True is optimized — faster than passing a mask tensor)
output_causal = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
print(f"Causal SDPA: {output_causal.shape}")

# With dropout (only during training)
output_dropout = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.1)

# SDPA backends explained:
print("""
SDPA Backends:
┌─────────────────────┬────────────────────────────────────────┐
│ Flash Attention     │ Fastest, O(N) memory, requires GPU     │
│ Efficient Attention │ Good memory efficiency, broader compat │  
│ Math (fallback)     │ Exact computation, works on CPU        │
└─────────────────────┴────────────────────────────────────────┘
On CPU, only the Math backend is used. On GPU, PyTorch auto-selects.
""")

In [ ]:
# Demonstrate: SDPA is equivalent to our manual implementation
torch.manual_seed(0)
B, H, T, D = 1, 1, 6, 16
Q = torch.randn(B, H, T, D)
K = torch.randn(B, H, T, D)
V = torch.randn(B, H, T, D)

# Manual implementation
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(D)
manual_out = F.softmax(scores, dim=-1) @ V

# SDPA implementation
sdpa_out = F.scaled_dot_product_attention(Q, K, V)

print(f"Manual vs SDPA max difference: {(manual_out - sdpa_out).abs().max().item():.2e}")
print("They compute the same thing — SDPA is just faster on GPU!")

## Building a Complete Transformer Block

A Transformer block combines:
1. **Multi-head self-attention** (with residual connection + LayerNorm)
2. **Feed-forward network** (with residual connection + LayerNorm)

This is the building block that gets stacked N times to build models like GPT, BERT, etc.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm Transformer block (used in GPT-2/3, LLaMA, etc.)."""
    
    def __init__(self, d_model, num_heads, ff_dim=None, dropout=0.1):
        super().__init__()
        ff_dim = ff_dim or 4 * d_model  # Standard: FFN is 4x wider
        
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm attention with residual
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, mask=mask)
        x = x + self.dropout(attn_out)
        
        # Pre-norm FFN with residual
        normed = self.norm2(x)
        ffn_out = self.ffn(normed)
        x = x + self.dropout(ffn_out)
        
        return x


# Test the Transformer block
d_model, num_heads = 128, 8
block = TransformerBlock(d_model, num_heads)

x = torch.randn(2, 20, d_model)
output = block(x)

print(f"Input:  {x.shape}")
print(f"Output: {output.shape} (same shape — blocks are stackable!)")
print(f"\nBlock parameters: {sum(p.numel() for p in block.parameters()):,}")
print(f"  Attention: {sum(p.numel() for p in block.attn.parameters()):,}")
print(f"  FFN:       {sum(p.numel() for p in block.ffn.parameters()):,}")
print(f"  LayerNorm: {sum(p.numel() for p in block.norm1.parameters()) + sum(p.numel() for p in block.norm2.parameters()):,}")

## Positional Encoding

Self-attention is permutation-equivariant — it doesn't know about position! We must inject position information. The original Transformer uses sinusoidal positional encodings:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

These give each position a unique fingerprint, and nearby positions have similar encodings.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# Visualize positional encodings
pe = SinusoidalPositionalEncoding(d_model=64, max_len=100)
encoding = pe.pe[0]  # [100, 64]

print(f"Positional encoding shape: {encoding.shape}")
print(f"\nPosition 0, first 8 dims: {encoding[0, :8]}")
print(f"Position 1, first 8 dims: {encoding[1, :8]}")
print(f"Position 99, first 8 dims: {encoding[99, :8]}")

# Show that nearby positions are more similar
cos_sim = F.cosine_similarity
pos0 = encoding[0:1]
sims = [F.cosine_similarity(pos0, encoding[i:i+1], dim=-1).item() for i in range(0, 100, 10)]
print(f"\nCosine similarity of position 0 with positions 0,10,20,...,90:")
print([f"{s:.3f}" for s in sims])
print("(Nearby positions are more similar!)")

## FlexAttention Overview

`torch.nn.attention.flex_attention` is PyTorch's API for expressing custom attention patterns without writing custom CUDA kernels. Instead of materializing a full attention mask, you define a **score_mod** function that modifies attention scores element-wise.

Key patterns:
- **Causal**: zero out future positions
- **Sliding window**: only attend to nearby positions
- **ALiBi**: linear position bias (no explicit positional encoding needed)
- **Document masking**: prevent cross-document attention in packed sequences

In [ ]:
# FlexAttention patterns (conceptual — actual API requires torch.compile + GPU)
# Here we show the pattern logic that FlexAttention uses

def causal_score_mod(score, b, h, q_idx, kv_idx):
    """Causal: mask future positions."""
    return torch.where(q_idx >= kv_idx, score, float('-inf'))

def sliding_window_score_mod(score, b, h, q_idx, kv_idx, window_size=128):
    """Sliding window: only attend within a local window."""
    return torch.where(
        (q_idx - kv_idx).abs() <= window_size,
        score,
        float('-inf')
    )

def alibi_score_mod(score, b, h, q_idx, kv_idx, num_heads=8):
    """ALiBi: add linear position bias (no positional encoding needed)."""
    slope = 2.0 ** (-(8.0 * (h + 1) / num_heads))
    bias = -slope * (q_idx - kv_idx).abs().float()
    return score + bias

# Demonstrate ALiBi bias pattern
seq_len = 8
biases = torch.zeros(seq_len, seq_len)
slope = 2.0 ** (-1.0)  # head 0 slope
for q in range(seq_len):
    for k in range(seq_len):
        biases[q, k] = -slope * abs(q - k)

print("ALiBi bias matrix (head 0) — penalizes distant positions:")
print(biases)
print("\nKey insight: no learned positional embeddings needed!")
print("ALiBi implicitly encodes position through attention score bias.")

# Note: actual flex_attention usage
print("""
# Production usage (requires GPU + torch.compile):
from torch.nn.attention.flex_attention import flex_attention, create_block_mask

def causal_mod(score, b, h, q_idx, kv_idx):
    return torch.where(q_idx >= kv_idx, score, -float('inf'))

block_mask = create_block_mask(causal_mod, B=1, H=1, Q_LEN=1024, KV_LEN=1024)
out = flex_attention(Q, K, V, score_mod=causal_mod, block_mask=block_mask)
""")

## Try It Yourself: Build a 2-Layer Transformer Encoder

**Exercise:** Combine the components we built to create a full 2-layer Transformer encoder. Your model should:
1. Accept token indices → embed them
2. Add positional encoding
3. Pass through 2 Transformer blocks
4. Test on random input and verify output shapes

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # x: [batch, seq_len] token indices
        h = self.embedding(x)       # [batch, seq, d_model]
        h = self.pos_encoding(h)    # add positional info
        
        for layer in self.layers:
            h = layer(h)
        
        return self.final_norm(h)


# Test the encoder
vocab_size = 1000
d_model = 128
num_heads = 8
num_layers = 2

encoder = TransformerEncoder(vocab_size, d_model, num_heads, num_layers)

# Random "token" input
tokens = torch.randint(0, vocab_size, (4, 32))  # batch=4, seq_len=32
output = encoder(tokens)

print(f"Input tokens: {tokens.shape}")
print(f"Encoder output: {output.shape}")
print(f"\nTotal parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  Embedding: {encoder.embedding.weight.numel():,}")
print(f"  Transformer layers: {sum(p.numel() for layer in encoder.layers for p in layer.parameters()):,}")

## Key Takeaways

1. **Attention = soft dictionary lookup** — queries search keys, retrieve weighted values
2. **Scaling by sqrt(d_k)** keeps gradients healthy as dimensions grow
3. **Causal masking** prevents future-peeking in autoregressive models
4. **Multi-head attention** lets the model learn diverse relationship patterns
5. **SDPA** (`F.scaled_dot_product_attention`) is the production API — auto-selects Flash/Efficient/Math backends
6. **Transformer block** = Attention + FFN + Residuals + LayerNorm — same in/out shape, so they stack
7. **Positional encodings** inject sequence order information (sinusoidal, learned, or relative like ALiBi)
8. **FlexAttention** lets you define custom attention patterns via `score_mod` without writing CUDA kernels
9. Pre-norm (norm before attention/FFN) is more stable than post-norm for deep networks
10. The FFN is typically 4x wider than d_model — it's where most parameters live